# 01 - Dataset Generation

Create a 50-question evaluation set from chunks already indexed in Mini RAG. This notebook expects the backend to be running, the project to contain ready documents, and `BENCHMARK_API_KEY`, `BENCHMARK_PROJECT_ID`, and `OPENAI_API_KEY` to be set in your environment.


## Cell 1 - Setup

Configure API access, OpenAI models, output paths, and helper functions used by later cells.


In [ ]:

import os, json, time, random, math, re, requests
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

# Configuration
API_BASE = os.getenv("BENCHMARK_API_BASE", "http://localhost:8000").rstrip("/")
API_KEY = os.getenv("BENCHMARK_API_KEY", "your-api-key-here")
PROJECT_ID = os.getenv("BENCHMARK_PROJECT_ID", "your-project-id-here")
OUTPUT_FILE = "results/test_dataset.json"

GENERATION_MODEL = os.getenv("BENCHMARK_GENERATION_MODEL", "gpt-3.5-turbo")
EMBEDDING_MODEL = os.getenv("BENCHMARK_EMBEDDING_MODEL", "text-embedding-3-small")
N_CHUNKS_TO_SAMPLE = int(os.getenv("BENCHMARK_N_CHUNKS", "25"))
RANDOM_SEED = int(os.getenv("BENCHMARK_RANDOM_SEED", "42"))
REQUEST_TIMEOUT_SECONDS = int(os.getenv("BENCHMARK_REQUEST_TIMEOUT_SECONDS", "60"))

Path("results").mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

client = OpenAI()
session = requests.Session()
session.headers.update({"X-API-Key": API_KEY, "Content-Type": "application/json"})


def api_get(path, params=None):
    url = f"{API_BASE}{path}"
    response = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    if not response.ok:
        try:
            detail = response.json().get("detail", response.text)
        except Exception:
            detail = response.text
        raise RuntimeError(f"GET {url} failed with {response.status_code}: {detail}")
    return response.json()


def require_configured():
    missing = []
    if API_KEY == "your-api-key-here":
        missing.append("BENCHMARK_API_KEY")
    if PROJECT_ID == "your-project-id-here":
        missing.append("BENCHMARK_PROJECT_ID")
    if missing:
        raise ValueError(f"Set these environment variables before running: {', '.join(missing)}")


require_configured()
print(f"Backend: {API_BASE}")
print(f"Project: {PROJECT_ID}")
print(f"Generation model: {GENERATION_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")


## Cell 2 - Load chunks from the project

Load all documents for the project, then fetch each document's chunks from the benchmark admin endpoint. The endpoint is authenticated and tenant-scoped.


In [ ]:

def list_project_documents(project_id):
    documents = []
    page = 1
    while True:
        payload = api_get("/api/documents", params={"project_id": project_id, "page": page, "per_page": 100})
        items = payload.get("items", payload if isinstance(payload, list) else [])
        documents.extend(items)
        total = payload.get("total", len(documents)) if isinstance(payload, dict) else len(documents)
        if len(documents) >= total or not items:
            break
        page += 1
    return documents


def load_document_chunks(project_id, document_id):
    payload = api_get(
        "/api/admin/chunks",
        params={"project_id": project_id, "document_id": document_id, "limit": 20000},
    )
    return payload.get("items", [])


documents = list_project_documents(PROJECT_ID)
chunks = []

for document in tqdm(documents, desc="Loading chunks"):
    doc_id = document["id"]
    doc_chunks = load_document_chunks(PROJECT_ID, doc_id)
    for chunk in doc_chunks:
        chunk.setdefault("document_id", doc_id)
        chunk.setdefault("document_name", document.get("original_filename", ""))
    chunks.extend(doc_chunks)

if not chunks:
    raise RuntimeError(
        "No chunks were loaded. Confirm documents are fully indexed and /api/admin/chunks is available."
    )

print(f"Loaded {len(chunks)} chunks from {len(documents)} documents")
pd.DataFrame(chunks)[["chunk_id", "document_id", "document_name", "chunk_index", "char_count"]].head()


## Cell 3 - Generate questions using GPT

Sample 25 chunks and ask the model for one easy and one hard question per chunk. Failed generations are retried and skipped only after repeated parsing errors.


In [ ]:

GENERATION_PROMPT = """You are creating a test dataset for evaluating a RAG system.

Given the following text excerpt from a document, generate exactly 2 specific questions
that satisfy ALL of these requirements:
1. The question MUST be answerable using ONLY this exact text
2. The question tests a specific fact (not general knowledge)
3. The question requires reading the text carefully to answer
4. Include both an "easy" and a "hard" question
5. The hard question requires synthesizing multiple sentences

Return ONLY valid JSON array, no other text:
[
  {{
    "question": "...",
    "answer": "exact quote or close paraphrase from text",
    "difficulty": "easy"
  }},
  {{
    "question": "...", 
    "answer": "...",
    "difficulty": "hard"
  }}
]

Text excerpt:
{text}"""


def extract_json_array(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON array found in response: {raw_text[:200]}")
    return json.loads(text[start : end + 1])


def generate_questions_for_chunk(chunk, max_retries=3):
    text = str(chunk.get("text", "")).strip()
    if len(text) > 4500:
        text = text[:4500]
    prompt = GENERATION_PROMPT.format(text=text)

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GENERATION_MODEL,
                messages=[
                    {"role": "system", "content": "You create precise JSON benchmark datasets."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
            )
            content = response.choices[0].message.content or ""
            items = extract_json_array(content)
            if not isinstance(items, list) or len(items) != 2:
                raise ValueError("Model did not return exactly two questions")

            normalized = []
            for item in items:
                question = str(item.get("question", "")).strip()
                answer = str(item.get("answer", "")).strip()
                difficulty = str(item.get("difficulty", "")).strip().lower()
                if difficulty not in {"easy", "hard"}:
                    raise ValueError(f"Invalid difficulty: {difficulty}")
                if not question or not answer:
                    raise ValueError("Question and answer are required")
                normalized.append({"question": question, "answer": answer, "difficulty": difficulty})
            return normalized
        except Exception as exc:
            last_error = exc
            time.sleep(1 + attempt)

    raise RuntimeError(f"Failed to generate questions for chunk {chunk.get('chunk_id')}: {last_error}")


sample_size = min(N_CHUNKS_TO_SAMPLE, len(chunks))
sampled_chunks = random.sample(chunks, sample_size)
questions = []
failures = []

for chunk in tqdm(sampled_chunks, desc="Generating Q&A pairs"):
    try:
        generated = generate_questions_for_chunk(chunk)
        for item in generated:
            questions.append(
                {
                    "id": f"q_{len(questions) + 1:04d}",
                    "question": item["question"],
                    "answer": item["answer"],
                    "difficulty": item["difficulty"],
                    "chunk_id": chunk.get("chunk_id") or chunk.get("id"),
                    "document_id": chunk.get("document_id"),
                    "document_name": chunk.get("document_name", ""),
                    "chunk_index": chunk.get("chunk_index"),
                    "source_text": chunk.get("text", ""),
                }
            )
    except Exception as exc:
        failures.append({"chunk_id": chunk.get("chunk_id") or chunk.get("id"), "error": str(exc)})
    time.sleep(0.5)

if failures:
    print(f"Skipped {len(failures)} chunks after generation failures")
    display(pd.DataFrame(failures).head())

print(f"Generated {len(questions)} questions from {sample_size} chunks")
pd.DataFrame(questions).head()


## Cell 4 - Deduplicate similar questions

Embed every generated question and remove near duplicates above cosine similarity `0.95`.


In [ ]:

def embed_texts(texts, batch_size=100):
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding questions"):
        batch = texts[start : start + batch_size]
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=batch)
        vectors.extend([item.embedding for item in response.data])
        time.sleep(0.2)
    return np.array(vectors, dtype=np.float32)


if not questions:
    raise RuntimeError("No questions generated; run Cell 3 successfully before deduplication.")

question_texts = [item["question"] for item in questions]
embeddings = embed_texts(question_texts)
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized = embeddings / np.clip(norms, 1e-12, None)
similarity = normalized @ normalized.T

keep_indices = []
duplicate_indices = set()

for i in range(len(questions)):
    if i in duplicate_indices:
        continue
    keep_indices.append(i)
    for j in range(i + 1, len(questions)):
        if similarity[i, j] > 0.95:
            duplicate_indices.add(j)

deduped_questions = [questions[i] for i in keep_indices]
for idx, item in enumerate(deduped_questions, start=1):
    item["id"] = f"q_{idx:04d}"

removed = len(questions) - len(deduped_questions)
questions = deduped_questions
print(f"Removed {removed} duplicates. Final dataset: {len(questions)} questions")


## Cell 5 - Manual review sample

Inspect a random sample and basic dataset statistics before saving.


In [ ]:

if not questions:
    raise RuntimeError("No questions available for review.")

review_df = pd.DataFrame(questions)
sample_n = min(5, len(review_df))
review_sample = review_df.sample(sample_n, random_state=RANDOM_SEED)[
    ["id", "difficulty", "question", "answer", "document_name", "chunk_id"]
]

display(review_sample)

print("Statistics")
print(f"- Total questions: {len(review_df)}")
print("- Easy/Hard breakdown:")
print(review_df["difficulty"].value_counts().to_string())
print(f"- Average question length: {review_df['question'].str.len().mean():.1f} characters")
print("- Questions per document:")
print(review_df["document_name"].fillna("unknown").value_counts().to_string())


## Cell 6 - Save dataset

Write JSON for benchmark execution and CSV for quick spreadsheet review.


In [ ]:

output_path = Path(OUTPUT_FILE)
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as file:
    json.dump(questions, file, indent=2, ensure_ascii=False)

csv_path = output_path.with_suffix(".csv")
pd.DataFrame(questions).to_csv(csv_path, index=False)

print("Dataset saved. Ready for benchmarking.")
print(f"JSON: {output_path}")
print(f"CSV:  {csv_path}")
